# AI/ML Business Use Case 

## ML Scoring for Proposal Development in the Consulting Industry

### Analysis and Report Presented by Pascal Chisom Okechukwu

## Exploratory Data Analysis for RFP ML Dataset

In [ ]:
# Importing the relevant libraries 

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Loading the dataset and overview

In [ ]:
# Dataset has previously been prepared 

rfp_df = pd.read_csv("prepared_rfp_dataset.csv")
rfp_df.info()

In [ ]:
# changing the [Award Date] to a datetime object 

rfp_df["Award Date"] = pd.to_datetime(rfp_df["Award Date"], errors='coerce')
rfp_df.info()

#### Basic overview

In [ ]:
print("Dataset Shape:", rfp_df.shape)
print("Columns:", rfp_df.columns.tolist())
print("\nClass Distribution:")
print(rfp_df["Proposal Status"].value_counts())

In [ ]:
rfp_df['Region'].value_counts()

In [ ]:
rfp_df['Subregion'].value_counts()

#### Visualising the [Proposal Status] distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=rfp_df, x="Proposal Status", palette="Set2")
plt.title("Proposal Status Distribution")
plt.tight_layout()
plt.show()

#### Visualizing the time trend of awards

In [ ]:
rfp_df['Year'] = rfp_df['Award Date'].dt.year
plt.figure(figsize=(10, 5))
sns.countplot(data=rfp_df, x="Year", hue="Proposal Status", palette="Set2")
plt.title("Proposals Over Time")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Average Award Amount by Status
plt.figure(figsize=(6, 4))
sns.boxplot(data=rfp_df, x="Proposal Status", y="Contract Award Amount (USD)", palette="Set2")
plt.title("Award Amount by Proposal Status")
plt.tight_layout()
plt.show()

In [ ]:
# Regional Distribution
plt.figure(figsize=(10, 5))
sns.countplot(data=rfp_df, y="Region", hue="Proposal Status", palette="Set2", order=rfp_df['Region'].value_counts().index)
plt.title("Proposals by Region")
plt.tight_layout()
plt.show()

#### Some Feature Engineering

There appears to be many unknowns in the geography data. We will use the following approach to address this: 
- Drop all records where [Country] is 'Unknown', as these would be hardest to manouvre.
- Creates a new binary column 'Has Known Country' based on the [Region], which will have 'No' for records with unknown region

In [ ]:
rfp_df = rfp_df[rfp_df['Country'] != 'Unknown'].copy()

# Create binary feature 'Has Known Country' based on whether Region is known
rfp_df['Has Known Country'] = rfp_df['Region'].apply(lambda x: 0 if x == 'Unknown' else 1)

# Confirm changes
print("After dropping 'Unknown' Country entries:")
print("Remaining records:", len(rfp_df))
print("Has Known Country distribution:")
print(rfp_df['Has Known Country'].value_counts())

In [ ]:
# Heatmap of correlation (if numeric features present)
numeric_df = rfp_df.select_dtypes(include='number')
if not numeric_df.empty:
    plt.figure(figsize=(8, 5))
    sns.heatmap(numeric_df.corr(), annot=True, cmap="Blues")
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

##### Saving to a new CSV for model development

In [ ]:
rfp_df.to_csv('rfp_business_case_data_clean.csv', index=False)

## Model Development and Evaluation

In [ ]:
# Importing the remaining relevant libraries 

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay

In [ ]:
df = rfp_df 
df.head(10)

In [ ]:
df.info()

In [ ]:
# Target and Features
features = [
    "Commodity Category", "WBG Organization", "Fund Source",
    "Region", "Contract Award Amount (USD)", "Has Known Country"
]
target = "Proposal Status"

X = df[features]
y = df[target].apply(lambda x: 1 if x == "Won" else 0)

In [ ]:
# Defining preprocessing
categorical_cols = ["Commodity Category", "WBG Organization", "Fund Source", "Region"]
numerical_cols = ["Contract Award Amount (USD)", "Has Known Country"]

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

preprocessor = ColumnTransformer([
    ("cat", categorical_transformer, categorical_cols),
    ("num", numerical_transformer, numerical_cols)
])

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

We will be using two models to train the data: Logistic regression and random forest 

#### Logistic Regression

In [ ]:
log_reg_pipeline = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=42))
log_reg_pipeline.fit(X_train, y_train)
y_pred_logreg = log_reg_pipeline.predict(X_test)
y_proba_logreg = log_reg_pipeline.predict_proba(X_test)[:, 1]
print("Logistic Regression Report:\n", classification_report(y_test, y_pred_logreg))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_logreg))

#### Random Forest 

In [ ]:
rf_pipeline = make_pipeline(preprocessor, RandomForestClassifier(n_estimators=100, random_state=42))
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]
print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

#### Confusion Matrix

In [ ]:
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(rf_pipeline, X_test, y_test, cmap="Blues", values_format='d')
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance Plot
feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_cols)
feature_names = np.append(feature_names, numerical_cols)
importances = rf_pipeline.named_steps['randomforestclassifier'].feature_importances_
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.barh(range(15), importances[sorted_idx][:15][::-1], align='center')
plt.yticks(range(15), feature_names[sorted_idx][:15][::-1])
plt.xlabel("Feature Importance")
plt.title("Top 15 Feature Importances - Random Forest")
plt.tight_layout()
plt.show()